This notebook converts a CSV file of [Apache Arrow community meeting](https://arrow.apache.org/community/#meetings) attendees into a compact [Arrow IPC](https://arrow.apache.org/docs/format/Columnar.html#serialization-and-interprocess-communication-ipc) file, demonstrating how careful choice of column types and compression can dramatically reduce file size—even in Arrow, which isn't usually known for being as compact as formats like Parquet that apply efficient encodings and compression more automatically.

## Setup

Install the required dependencies:

In [1]:
%pip install -q pyarrow

Note: you may need to restart the kernel to use updated packages.


Import PyArrow and its modules:

In [2]:
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as csv
import pyarrow.ipc as ipc

## The Dataset

The dataset lists the attendees of all the biweekly Apache Arrow community meetings from January 18, 2023 through March 25, 2026. Meeting notes are captured in a public [Google Doc](https://docs.google.com/document/d/1xrji8fc6_24TVmKiHJB4ECX1Zy2sy2eRbBjpVJMnPmk/). The Google Doc was saved to a Markdown file, and then Claude Code was used to extract the meeting dates and attendee names, fix misspellings, and canonicalize names where nicknames were used. The result is `attendees.csv`: one row per attendee per meeting, with `date` and `name` columns.

Read the CSV file into a PyArrow table:

In [3]:
data_dir = "../data/arrow-meeting-attendees"
convert_options = csv.ConvertOptions(column_types={"date": pa.date32(), "name": pa.string()})
in_table = csv.read_csv(f"{data_dir}/attendees.csv", convert_options=convert_options)
in_table

pyarrow.Table
date: date32[day]
name: string
----
date: [[2026-03-25,2026-03-25,2026-03-25,2026-03-25,2026-03-25,...,2023-01-18,2023-01-18,2023-01-18,2023-01-18,2023-01-18]]
name: [["Ian Cook","Adam Reeve","Martin Prammer","Raúl Cumplido","Pedro Matias",...,"David Li","Bryce Mecum","Rok Mihevc","Matt Topol","Jacob Wujciak-Jens"]]

## Encoding the Columns

The key insight is to match the encoding to the structure of the data:

- **Run-end encoding for `date`**: The CSV is sorted by meeting date, so all attendees of a given meeting appear in a contiguous run with the same date value. [Run-end encoding](https://arrow.apache.org/docs/format/Columnar.html#run-end-encoded-layout) stores each run as a single (value, run-end) pair rather than repeating the value for every row. With ~10–20 attendees per meeting, this is a huge win. We use `int16` for the run-end index type since the dataset is small.
- **Dictionary encoding for `name`**: The same people tend to show up to these meetings over and over. [Dictionary encoding](https://arrow.apache.org/docs/format/Columnar.html#dictionary-encoded-layout) stores each unique name once in a dictionary and replaces every occurrence with a small integer index. This avoids repeating long strings.

Apply the encodings to each column:

In [4]:
ree_dates = pc.run_end_encode(in_table.column("date"), run_end_type=pa.int16())
dict_names = pc.dictionary_encode(in_table.column("name"))

Assemble the encoded columns into an output table:

In [5]:
out_table = pa.Table.from_arrays([ree_dates, dict_names], names=["date", "name"])
out_table

pyarrow.Table
date: run_end_encoded<run_ends: int16, values: date32[day]>
  child 0, run_ends: int16 not null
  child 1, values: date32[day]
name: dictionary<values=string, indices=int32, ordered=0>
----
date: [  -- run_ends:
[8,21,33,41,53,...,921,932,944,957,969]  -- values:
[2026-03-25,2026-03-11,2026-02-25,2026-02-11,2026-01-28,...,2023-03-15,2023-03-01,2023-02-15,2023-02-01,2023-01-18]]
name: [  -- dictionary:
["Ian Cook","Adam Reeve","Martin Prammer","Raúl Cumplido","Pedro Matias",...,"Gang Wu","Weibin Zeng","Ian Joiner","Anja Boskovic","Sean Kelly"]  -- indices:
[0,1,2,3,4,...,69,9,10,14,33]]

## Writing with Compression

Write the output as an Arrow IPC file with [Arrow IPC compression](https://arrow.apache.org/docs/format/Columnar.html#compression) using the [zstd codec](https://facebook.github.io/zstd/), which squeezes out additional savings on top of the efficient encodings:

In [6]:
write_options = ipc.IpcWriteOptions(compression="zstd")
with pa.OSFile(f"{data_dir}/attendees.arrow", "wb") as sink:
    with ipc.new_file(sink, out_table.schema, options=write_options) as writer:
        writer.write_table(out_table)

## Results

How much smaller is the Arrow IPC file compared to the original CSV?

In [7]:
import os

csv_size = os.path.getsize(f"{data_dir}/attendees.csv")
arrow_size = os.path.getsize(f"{data_dir}/attendees.arrow")
print(f"attendees.csv:   {csv_size:,} bytes")
print(f"attendees.arrow: {arrow_size:,} bytes")
print(f"Reduction:       {1 - arrow_size / csv_size:.0%} smaller")

attendees.csv:   23,388 bytes
attendees.arrow: 3,674 bytes
Reduction:       84% smaller


## Trade-offs

There's no free lunch here. Compression and specialized encodings like run-end encoding (REE) and dictionary encoding add processing overhead—CPU time to compress and decompress, and potentially casting/conversion time at the destination if it doesn't natively support some newer Arrow types like REE. Whether these trade-offs are worth it depends on the bottleneck:

- **Fast network, low latency requirements**: If you're transmitting data over a very fast local network or shared memory, the compression and decompression overhead may cost more than the time saved by sending fewer bytes. In these cases, it can make sense to skip compression and choose plain types that match the origin and destination formats to minimize processing on both ends.
- **Slower network or extremely high throughput**: When bandwidth is the bottleneck—slower networks, cross-region transfers, or very high-throughput pipelines where every byte counts—the CPU cost of compression and encoding pays for itself many times over in reduced transfer time and I/O.

The right choice depends on your specific workload. Profile it.

## Try It Yourself

Some things to experiment with:

- **Turn off compression** by removing the `compression` option from `IpcWriteOptions`. How much of the size reduction comes from the encodings alone vs. compression?
- **Switch to lz4 compression** (`compression="lz4"`) instead of zstd. How does the file size and write speed compare?
- **Change the column types**: What happens if you skip the run-end encoding and just use a plain `date32` column? What if you skip dictionary encoding for names?
- **Change the run-end index type**: We used `int16`—try `int32` or `int64` and see the effect.

You can also use any Arrow-compatible tool to analyze the data itself. For example, who attends these meetings the most? Load `attendees.arrow` in PyArrow, DuckDB, Polars, or DataFusion and find out.